In [23]:
%pip install --upgrade pip setuptools wheel


Note: you may need to restart the kernel to use updated packages.


In [24]:
%pip install -U \
    category-encoders \
    # cffi==1.16.0 \
    cloudpickle==3.0.0 \
    nltk==3.9.2 \
    defusedxml==0.7.1 \
    graphviz==0.20.3 \
    holidays==0.54 \
    lightgbm==4.5.0 \
    # lz4==4.3.3 \
    matplotlib==3.9.2 \
    psutil==5.9.8 \
    pyarrow==15.0.2 \
    optuna


Note: you may need to restart the kernel to use updated packages.


In [25]:
%pip install optuna 

Note: you may need to restart the kernel to use updated packages.


In [26]:
import pandas as pd
train = pd.read_csv(
    "/Users/saurabh.prajapati/Documents/Medisyn-Labs/data/train.tsv",
    sep='\t',
    names=[
        "customer_identifier",
        "medicine_name",
        "rating",
        "effectiveness",
        "side_effects",
        "illness",
        "review_benefits",
        "review_sideEffects",
        "review_overall"
    ],
    header=None
)
train['positive'] = (train['rating'] > 6).astype(int)

In [27]:
import pandas as pd
test = pd.read_csv(
    "/Users/saurabh.prajapati/Documents/Medisyn-Labs/data/test.tsv",
    sep='\t',
    names=[
        "customer_identifier",
        "medicine_name",
        "rating",
        "effectiveness",
        "side_effects",
        "illness",
        "review_benefits",
        "review_sideEffects",
        "review_overall"
    ],
    header=None
)
test['positive'] = (test['rating'] > 6).astype(int)

In [28]:
review_template = (
    "review_benefits: {review_benefits}\n"
    "review_sideEffects: {review_sideEffects}\n"
    "review_overall: {review_overall}  \n"
    # "effectiveness: {effectiveness}  \n"
    "medicine_name: {medicine_name}  \n"
    # "side_effects: {side_effects} \n"
    "illness: {illness}  "
)

train['review'] = train.apply(lambda row: review_template.format(**row), axis=1)
test['review'] = test.apply(lambda row: review_template.format(**row), axis=1)

In [29]:
df=pd.concat([train,test],axis=0)
df.drop(['rating','customer_identifier','review'],axis=1,inplace=True)

In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4143 entries, 0 to 1035
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   medicine_name       4143 non-null   object
 1   effectiveness       4143 non-null   object
 2   side_effects        4143 non-null   object
 3   illness             4142 non-null   object
 4   review_benefits     4120 non-null   object
 5   review_sideEffects  4045 non-null   object
 6   review_overall      4130 non-null   object
 7   positive            4143 non-null   int64 
dtypes: int64(1), object(7)
memory usage: 291.3+ KB


In [31]:
df.isnull().sum()

medicine_name          0
effectiveness          0
side_effects           0
illness                1
review_benefits       23
review_sideEffects    98
review_overall        13
positive               0
dtype: int64

In [32]:
train.shape

(3107, 11)

In [33]:
target_col='positive'
nlp_col=['review_benefits','review_sideEffects','review_overall']
boolean_cols = df.select_dtypes(include="boolean").columns
numerical_cols = df.select_dtypes(include="number").columns
numerical_cols = list(set(numerical_cols)-set([target_col]))
categorical_cols = df.select_dtypes(include="object").columns
categorical_cols=list(set(categorical_cols)-set(nlp_col))

summary = {
    "boolean_cols": boolean_cols,
    "numerical_cols": numerical_cols,
    "categorical_cols": categorical_cols
}

for col_type, cols in summary.items():
    print(f"{col_type} ({len(cols)} columns):")
    for col in cols:
        print(f" - {col}")
    print()

boolean_cols (0 columns):

numerical_cols (0 columns):

categorical_cols (4 columns):
 - effectiveness
 - side_effects
 - medicine_name
 - illness



In [34]:
cardinality = {"low": [], "medium": [], "high": []}

for col in categorical_cols:
    unique_count = df[col].nunique()
    if unique_count < 10:
        cardinality["low"].append((col, unique_count))
    elif 10 <= unique_count <= 50:
        cardinality["medium"].append((col, unique_count))
    else:
        cardinality["high"].append((col, unique_count))

for key, value in cardinality.items():
    print(f"{key.capitalize()} cardinality ({len(value)} columns):")
    for col, count in value:
        print(f" - {col}: {count}")
    print()
    
low_cardinal_cat_feats = [col for col, _ in cardinality['low']]
meidum_cardinal_cat_feats = [col for col, _ in cardinality['medium']]
high_cardinal_cat_feats = [col for col, _ in cardinality['high']]

Low cardinality (2 columns):
 - effectiveness: 5
 - side_effects: 5

Medium cardinality (0 columns):

High cardinality (2 columns):
 - medicine_name: 541
 - illness: 1807



In [35]:
import ssl
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context


In [36]:
import numpy as np
import pandas as pd
import re, emoji, nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tqdm.notebook import tqdm
from category_encoders import CatBoostEncoder, TargetEncoder
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# ----------------------------------------------------------------
# 🔹 1. Download NLTK resources (one-time)
# ----------------------------------------------------------------

import nltk
nltk.download('stopwords')
nltk.download('wordnet')

# ----------------------------------------------------------------
# 🔹 2. Stopwords and Lemmatizer setup
# ----------------------------------------------------------------
additional_stopwords = {
    "drug", "medicine", "tablet", "doctor", "patient", "review_benefits", "review_sideEffects",
    "review_overall", "mg", "day", "take", "took", "used", "taking", "pill", "one", "review",
    "dose", "dosage", "medication", "treatment", "therapy", "prescribed", "prescription", 
    "prescribe", "physician", "med", "antibiotic", "cream", "application", "apply", "use", 
    "using", "take", "taken", "stop", "stopped", "started", "starting", "start", "continue",
    "continued", "course", "treat", "treated", "treating", "dos", "mcg", "tab", "daily", 
    "nightly", "bedtime", "morning", "evening", "hour", "per", "pm", "qd", "weekly", "month",
    "week", "year", "night", "time", "two", "three", "four", "five", "every", "twice", "long",
    "within", "since", "around", "last", "next", "ago", "short",
    "january", "february", "march", "april", "may", "june", "july", "august", "september",
    "october", "november", "december", "jan", "feb", "mar", "apr", "jun", "jul", "aug", "sep",
    "sept", "oct", "nov", "dec"
}

stop_words = set(stopwords.words('english')).union(additional_stopwords)
lemmatizer = WordNetLemmatizer()

# ----------------------------------------------------------------
# 🔹 3. Text Preprocessor Function
# ----------------------------------------------------------------
def preprocess_text_fast(text):
    if not isinstance(text, str):
        return ""
    text = emoji.demojize(text, language='en')
    text = text.lower().strip()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    words = [lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words]
    return " ".join(words)

def nlp_text(X):
    X = X.copy()
    for col in X.columns:
        X[col] = X[col].apply(preprocess_text_fast)
    return X

# Updated cleaner to return Series
def nlp_text_series(x):
    if isinstance(x, pd.DataFrame):
        x = x.iloc[:, 0]  # Take first (only) column
    return x.apply(preprocess_text_fast)

# ----------------------------------------------------------------
# 🔹 4. Feature Groups
# ----------------------------------------------------------------

# ----------------------------------------------------------------
# 🔹 5. OneHotEncoder Pipeline (low-cardinality)
# ----------------------------------------------------------------
one_hot_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
CatBoostEncoder._get_tags = lambda self: {"allow_nan": True}
# ----------------------------------------------------------------
# 🔹 6. CatBoostEncoder Pipeline (high-cardinality)
# ----------------------------------------------------------------
catboost_pipeline = Pipeline([
    ("imputer", SimpleImputer(fill_value="NOT_AVAILABLE", strategy="constant")),
    ("encoder", CatBoostEncoder()),
    ("scaler", StandardScaler())
])

# ----------------------------------------------------------------
# 🔹 7. NLP Transformer (text column)
# ----------------------------------------------------------------
nlp_pipeline = Pipeline([
    ("cleaner", FunctionTransformer(nlp_text_series, validate=False)),
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1,2)))
])

# ----------------------------------------------------------------
# 🔹 8. Combine All Transformers
# ----------------------------------------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("one_hot", one_hot_pipeline, low_cardinal_cat_feats),
        ("catboost", catboost_pipeline, high_cardinal_cat_feats),
        ("nlp", nlp_pipeline, nlp_col),
    ],
    remainder="passthrough",
    sparse_threshold=1
)


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/saurabh.prajapati/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/saurabh.prajapati/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [37]:
req_cols=["medicine_name","effectiveness","side_effects","illness"]+ nlp_col

X_train, X_test, y_train, y_test = train[req_cols], test[req_cols],train[target_col],test[target_col]

In [38]:
summary = {
    "name": ["Train Set","Test Set"],
    "# samples": [X_train.shape[0], X_test.shape[0]],
    "positive_rate": [
        round(100 * y_train.sum() / y_train.shape[0], 2),
        round(100 * y_test.sum() / y_test.shape[0], 2)
    ]
}
pd.DataFrame(summary)

,name,# samples,positive_rate
0,Train Set,3107,68.55
1,Test Set,1036,64.67


In [39]:
import optuna

In [40]:

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_validate, StratifiedKFold
from lightgbm import LGBMClassifier
import optuna

# Define the objective function for Optuna
def objective(trial):
    param = {
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 10.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 10.0),
        "learning_rate": trial.suggest_float("learning_rate", 0.1, 0.5),
        "max_bin": trial.suggest_int("max_bin", 200, 500),
        "max_depth": trial.suggest_int("max_depth", 6, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 100),
        "n_estimators": trial.suggest_int("n_estimators", 5, 100),
        "num_leaves": trial.suggest_int("num_leaves", 20, 50),
        "path_smooth": trial.suggest_float("path_smooth", 0.0, 100.0),
        "random_state": 537672287,
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
    }

    model = Pipeline(
        [
            ("preprocessor", preprocessor),
            ("classifier", LGBMClassifier(**param)),
        ]
    )

    cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=427)
    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="f1",
        return_train_score=True,
        n_jobs=-1,
    )
    return cv_results["test_score"].mean()

# Create a study and optimize the objective function
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=5)

# Get the best parameters
best_params = study.best_params
best_params["random_state"] = 537672287


model = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", LGBMClassifier(**best_params)),
    ]
)

# Fit the model
model.fit(X_train, y_train)

# Stratified K-Fold for handling imbalanced classes
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=427)

# Evaluate model with cross-validation
cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=["accuracy", "precision", "recall", "f1", "roc_auc"],
    return_train_score=True,
    n_jobs=-1
)

[I 2025-11-02 12:41:12,816] A new study created in memory with name: no-name-14d93160-2ee5-45fb-8204-f531d075a00f
/Users/saurabh.prajapati/Documents/Medisyn-Labs/.venv/lib/python3.14/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] lambda_l1 is set=7.444306725408964, reg_alpha=0.0 will be ignored. Current value: lambda_l1=7.444306725408964
[LightGBM] [Warning] lambda_l2 is set=9.443140144593194, reg_lambda=0.0 will be ignored. Current value: lambda_l2=9.443140144593194
[LightGBM] [Warning] lambda_l1 is set=7.444306725408964, reg_alpha=0.0 will be ignored. Current value: lambda_l1=7.444306725408964
[LightGBM] [Warning] lambda_l2 is set=9.443140144593194, reg_lambda=0.0 will be ignored. Current value: lambda_l2=9.443140144593194
[LightGBM] [Info] Number of positive: 1065, number of negative: 488
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002048 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1898
[LightGBM] [Info] Number of data points in the train set: 1553, number of used features: 54
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.6857

/Users/saurabh.prajapati/Documents/Medisyn-Labs/.venv/lib/python3.14/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] lambda_l1 is set=7.444306725408964, reg_alpha=0.0 will be ignored. Current value: lambda_l1=7.444306725408964
[LightGBM] [Warning] lambda_l2 is set=9.443140144593194, reg_lambda=0.0 will be ignored. Current value: lambda_l2=9.443140144593194


/Users/saurabh.prajapati/Documents/Medisyn-Labs/.venv/lib/python3.14/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


...

/Users/saurabh.prajapati/Documents/Medisyn-Labs/.venv/lib/python3.14/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/saurabh.prajapati/Documents/Medisyn-Labs/.venv/lib/python3.14/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] lambda_l1 is set=6.9238838674616074, reg_alpha=0.0 will be ignored. Current value: lambda_l1=6.9238838674616074
[LightGBM] [Warning] lambda_l2 is set=9.707557167979227, reg_lambda=0.0 will be ignored. Current value: lambda_l2=9.707557167979227
[LightGBM] [Warning] lambda_l1 is set=6.9238838674616074, reg_alpha=0.0 will be ignored. Current value: lambda_l1=6.9238838674616074
[LightGBM] [Warning] lambda_l2 is set=9.707557167979227, reg_lambda=0.0 will be ignored. Current value: lambda_l2=9.707557167979227


/Users/saurabh.prajapati/Documents/Medisyn-Labs/.venv/lib/python3.14/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/saurabh.prajapati/Documents/Medisyn-Labs/.venv/lib/python3.14/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] lambda_l1 is set=6.9238838674616074, reg_alpha=0.0 will be ignored. Current value: lambda_l1=6.9238838674616074
[LightGBM] [Warning] lambda_l2 is set=9.707557167979227, reg_lambda=0.0 will be ignored. Current value: lambda_l2=9.707557167979227
[LightGBM] [Warning] lambda_l1 is set=6.9238838674616074, reg_alpha=0.0 will be ignored. Current value: lambda_l1=6.9238838674616074
[LightGBM] [Warning] lambda_l2 is set=9.707557167979227, reg_lambda=0.0 will be ignored. Current value: lambda_l2=9.707557167979227
[LightGBM] [Warning] lambda_l1 is set=6.9238838674616074, reg_alpha=0.0 will be ignored. Current value: lambda_l1=6.9238838674616074
[LightGBM] [Warning] lambda_l2 is set=9.707557167979227, reg_lambda=0.0 will be ignored. Current value: lambda_l2=9.707557167979227
[LightGBM] [Warning] lambda_l1 is set=6.9238838674616074, reg_alpha=0.0 will be ignored. Current value: lambda_l1=6.9238838674616074
[LightGBM] [Warning] lambda_l2 is set=9.707557167979227, reg_lambda=0.0 

/Users/saurabh.prajapati/Documents/Medisyn-Labs/.venv/lib/python3.14/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/saurabh.prajapati/Documents/Medisyn-Labs/.venv/lib/python3.14/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [41]:
from helper import *

best_proba_threshold = get_proba_threshold(model, X_test, y_test)

eval_metrics=log_model_eval_metrics(model, X_train, y_train, X_test, y_test, best_proba_threshold)

/Users/saurabh.prajapati/Documents/Medisyn-Labs/.venv/lib/python3.14/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] lambda_l1 is set=6.9238838674616074, reg_alpha=0.0 will be ignored. Current value: lambda_l1=6.9238838674616074
[LightGBM] [Warning] lambda_l2 is set=9.707557167979227, reg_lambda=0.0 will be ignored. Current value: lambda_l2=9.707557167979227
[LightGBM] [Warning] lambda_l1 is set=6.9238838674616074, reg_alpha=0.0 will be ignored. Current value: lambda_l1=6.9238838674616074
[LightGBM] [Warning] lambda_l2 is set=9.707557167979227, reg_lambda=0.0 will be ignored. Current value: lambda_l2=9.707557167979227
[LightGBM] [Warning] lambda_l1 is set=6.9238838674616074, reg_alpha=0.0 will be ignored. Current value: lambda_l1=6.9238838674616074
[LightGBM] [Warning] lambda_l2 is set=9.707557167979227, reg_lambda=0.0 will be ignored. Current value: lambda_l2=9.707557167979227
{'f1_score': [0.93, 0.92], 'precision': [0.91, 0.89], 'recall': [0.95, 0.94], 'roc_auc': [0.96, 0.95]}


/Users/saurabh.prajapati/Documents/Medisyn-Labs/.venv/lib/python3.14/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/saurabh.prajapati/Documents/Medisyn-Labs/.venv/lib/python3.14/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [42]:
eval_metrics

,type,f1_score,precision,recall,roc_auc
0,train,0.93,0.91,0.95,0.96
1,test,0.92,0.89,0.94,0.95


In [43]:
import numpy as np
train_ks_table, test_ks_table=apply_model_and_get_ks_table(model, X_train, y_train, X_test, y_test,  best_proba_threshold, y_true_col='positive', y_pred_proba_col='proba', verbose=True)

/Users/saurabh.prajapati/Documents/Medisyn-Labs/.venv/lib/python3.14/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/saurabh.prajapati/Documents/Medisyn-Labs/.venv/lib/python3.14/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] lambda_l1 is set=6.9238838674616074, reg_alpha=0.0 will be ignored. Current value: lambda_l1=6.9238838674616074
[LightGBM] [Warning] lambda_l2 is set=9.707557167979227, reg_lambda=0.0 will be ignored. Current value: lambda_l2=9.707557167979227
[LightGBM] [Warning] lambda_l1 is set=6.9238838674616074, reg_alpha=0.0 will be ignored. Current value: lambda_l1=6.9238838674616074
[LightGBM] [Warning] lambda_l2 is set=9.707557167979227, reg_lambda=0.0 will be ignored. Current value: lambda_l2=9.707557167979227
KS is 76.81% at Decile 6
KS is 74.25% at Decile 6


TypeError: cannot unpack non-iterable NoneType object

In [ ]:

# from helper import *
# from lightgbm import LGBMClassifier
# from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# from sklearn.model_selection import cross_validate, StratifiedKFold
# from lightgbm import LGBMClassifier
# import mlflow
# import matplotlib.pyplot as plt
# import warnings
# import optuna
# import shap
# from shap import KernelExplainer, summary_plot
# from optuna.integration import LightGBMPruningCallback

# # Define the objective function for Optuna
# def objective(trial):
#     param = {
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
#         "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 10.0),
#         "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 10.0),
#         "learning_rate": trial.suggest_float("learning_rate", 0.1, 0.5),
#         "max_bin": trial.suggest_int("max_bin", 200, 500),
#         "max_depth": trial.suggest_int("max_depth", 6, 16),
#         "min_child_samples": trial.suggest_int("min_child_samples", 20, 100),
#         "n_estimators": trial.suggest_int("n_estimators", 5, 100),
#         "num_leaves": trial.suggest_int("num_leaves", 20, 50),
#         "path_smooth": trial.suggest_float("path_smooth", 0.0, 100.0),
#         "random_state": 537672287,
#         "subsample": trial.suggest_float("subsample", 0.5, 1.0),
#     }

#     model = Pipeline(
#         [
#             ("preprocessor", preprocessor),
#             ("classifier", LGBMClassifier(**param)),
#         ]
#     )

#     cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=427)
#     cv_results = cross_validate(
#         model,
#         X_train,
#         y_train,
#         cv=cv,
#         scoring="f1",
#         return_train_score=True,
#         n_jobs=-1,
#     )
#     return cv_results["test_score"].mean()

# # Create a study and optimize the objective function
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=5)

# # Get the best parameters
# best_params = study.best_params
# best_params["random_state"] = 537672287
# # best_params["class_weight"] = class_weight_dict

# with mlflow.start_run(run_name="lightGBM_categorical_encoding_optuna_optimize_f1_score") as run:
#     # Define the model pipeline with best parameters
#     model = Pipeline(
#         [
#             ("preprocessor", preprocessor),
#             ("classifier", LGBMClassifier(**best_params)),
#         ]
#     )

#     # Fit the model
#     model.fit(X_train, y_train)

#     # Stratified K-Fold for handling imbalanced classes
#     cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=427)

#     # Evaluate model with cross-validation
#     cv_results = cross_validate(
#         model,
#         X_train,
#         y_train,
#         cv=cv,
#         scoring=["accuracy", "precision", "recall", "f1", "roc_auc"],
#         return_train_score=True,
#         n_jobs=-1
#     )

#     # Wrap the model for MLflow
#     wrapped_model = SklearnModelWrapper(model)
#     df = pd.concat([X_train, y_train], axis=1)
#     # Sample data for model signature
#     df_sample = df.sample(frac=0.01)
#     x_sample = df_sample.drop(columns=["positive"])
#     y_sample = df_sample["positive"].values.ravel()
#     signature = infer_signature(x_sample, model.predict(x_sample))
#     mlflow.pyfunc.log_model(
#         "lgbm_model", python_model=wrapped_model, signature=signature
#     )

#     # Log cross-validation metrics
#     mlflow.log_metrics({f"cv_{key}": value.mean() for key, value in cv_results.items()})
#     mlflow.log_params(best_params)
#     # Determine the best probability threshold
#     best_proba_threshold = get_proba_threshold(model, X_test, y_test)
#     mlflow.log_metric("best_proba_threshold", best_proba_threshold)
#     # Log data used for modeling
#     log_data_used_for_modelling(df, X_train, y_train, X_test, y_test)

#     # Log model evaluation metrics
#     log_model_eval_metrics(model, X_train, y_train, X_test, y_test, best_proba_threshold)


#     # Log KS tables
#     apply_model_and_get_ks_table(model, X_train, y_train, X_test, y_test,  best_proba_threshold, y_true_col='positive', y_pred_proba_col='proba', verbose=True)

#     # Sample background data for SHAP Explainer
#     train_sample = X_train.sample(n=min(100, X_train.shape[0]), random_state=484006742)
#     example = X_train.sample(n=min(100, X_train.shape[0]), random_state=484006742)

#     # Use Kernel SHAP to explain feature importance
#     predict = lambda x: model.predict(pd.DataFrame(x, columns=X_train.columns))
#     explainer = KernelExplainer(predict, train_sample, link="identity")
#     shap_values = explainer.shap_values(example, l1_reg=False, nsamples=100)

#     # Save and log SHAP summary plot
#     plt.figure()
#     summary_plot(shap_values, example, show=False)
#     plt.savefig("shap_summary_plot.png")
#     mlflow.log_artifact("shap_summary_plot.png")

#     # Set MLflow tag for the algorithm used
#     mlflow.set_tag(key='ml_algorithm', value='lgbm')